# Student Performance AI Analytics
## IBM SkillsBuild Data Analytics with AI Academic Internship Project

**Author:** Vinit  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn, Jupyter Notebook


## 1. Problem Statement
Educational institutions collect information such as attendance, study time, assignment performance, previous marks, and lifestyle patterns. This project analyzes these factors and builds a machine-learning model to predict final student performance.

## 2. Objectives
- Clean and prepare student performance data.
- Perform exploratory data analysis and visualization.
- Identify relationships between academic/lifestyle factors and final scores.
- Build and evaluate a machine-learning regression model.
- Analyze feature importance and generate practical recommendations.

## 3. Dataset
This project uses a synthetic dataset created for academic demonstration.

**Rows:** 250  
**Columns:** 7

Features include study hours, attendance, assignment score, previous score, sleep hours, internet access, and final score.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
df = pd.read_csv("student_performance_dataset.csv")
df.head()


## 4. Data Inspection and Cleaning

In [ ]:
print("Shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().copy()
df.describe(include="all")


## 5. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.histplot(df["Final_Score"], kde=True, ax=axes[0])
axes[0].set_title("Distribution of Final Scores")
sns.scatterplot(data=df, x="Study_Hours_Per_Day", y="Final_Score", ax=axes[1])
axes[1].set_title("Study Hours vs Final Score")
plt.tight_layout()
plt.show()


In [ ]:
numeric_cols=["Study_Hours_Per_Day","Attendance_Percent","Assignment_Score","Previous_Score","Sleep_Hours","Final_Score"]
plt.figure(figsize=(9,6))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x="Internet_Access", y="Final_Score")
plt.title("Final Score by Internet Access")
plt.show()


## 6. Feature Preparation

In [ ]:
X=df.drop("Final_Score",axis=1)
y=df["Final_Score"]
categorical_features=["Internet_Access"]
numeric_features=[c for c in X.columns if c not in categorical_features]
preprocessor=ColumnTransformer([
    ("num","passthrough",numeric_features),
    ("cat",OneHotEncoder(handle_unknown="ignore"),categorical_features)
])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,random_state=42)
print("Training rows:",len(X_train),"Testing rows:",len(X_test))


## 7. Machine Learning Model
A **Random Forest Regressor** is used because it can capture nonlinear relationships and interactions between multiple features.

In [ ]:
model=Pipeline([
    ("preprocessor",preprocessor),
    ("regressor",RandomForestRegressor(n_estimators=250,random_state=42,max_depth=10))
])
model.fit(X_train,y_train)
predictions=model.predict(X_test)
mae=mean_absolute_error(y_test,predictions)
rmse=np.sqrt(mean_squared_error(y_test,predictions))
r2=r2_score(y_test,predictions)
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.3f}")


In [ ]:
results=pd.DataFrame({"Actual_Score":y_test.values,"Predicted_Score":np.round(predictions,1)})
results.head(10)


In [ ]:
plt.figure(figsize=(7,6))
sns.scatterplot(x=y_test,y=predictions)
plt.plot([y_test.min(),y_test.max()],[y_test.min(),y_test.max()],linestyle="--")
plt.xlabel("Actual Score"); plt.ylabel("Predicted Score")
plt.title("Actual vs Predicted Final Score")
plt.show()


## 8. Feature Importance

In [ ]:
feature_names=numeric_features+list(model.named_steps["preprocessor"].named_transformers_["cat"].get_feature_names_out(categorical_features))
importance_df=pd.DataFrame({"Feature":feature_names,"Importance":model.named_steps["regressor"].feature_importances_}).sort_values("Importance",ascending=False)
display(importance_df)
plt.figure(figsize=(9,5))
sns.barplot(data=importance_df,x="Importance",y="Feature")
plt.title("Random Forest Feature Importance")
plt.tight_layout(); plt.show()


## 9. AI-Oriented Recommendations
The following rule-based component converts student data into simple academic-support suggestions.

In [ ]:
def performance_category(score):
    if score >= 85: return "Excellent"
    if score >= 70: return "Good"
    if score >= 50: return "Needs Improvement"
    return "At Risk"

def generate_recommendation(row):
    actions=[]
    if row["Attendance_Percent"]<75: actions.append("improve attendance")
    if row["Study_Hours_Per_Day"]<4: actions.append("increase focused study time")
    if row["Assignment_Score"]<70: actions.append("complete assignments more consistently")
    if row["Previous_Score"]<60: actions.append("revise fundamentals and previous topics")
    if row["Sleep_Hours"]<6: actions.append("maintain a healthier sleep schedule")
    return ("Recommended actions: "+", ".join(actions)+".") if actions else "Maintain the current study routine and continue regular revision."

sample=df.iloc[0]
print("Performance category:",performance_category(sample["Final_Score"]))
print(generate_recommendation(sample))


## 10. Key Findings
- Academic and lifestyle variables can be analyzed together to understand performance patterns.
- Study time, attendance, assignment performance, and previous scores are useful predictive variables.
- Random Forest provides a practical baseline for prediction.
- Feature importance helps explain which variables contribute to predictions.

## 11. Limitations
- The dataset is synthetic and intended for educational demonstration.
- Real student behavior may involve additional variables.
- Correlation does not prove causation.
- Model performance may not generalize to another population.

## 12. Future Scope
- Use a larger real-world dataset with privacy safeguards.
- Add historical/time-series academic records.
- Compare additional ML algorithms.
- Build a Streamlit dashboard.
- Add explainable AI such as SHAP.

## 13. Conclusion
This project demonstrates how data analytics and machine learning can be combined to understand and predict student performance. It provides a foundation for an academic analytics dashboard that can support evidence-based student assistance.